In [9]:
import h5py

def walk_h5(g, prefix=""):
    for key in g.keys():
        item = g[key]
        path = f"{prefix}/{key}"
        if isinstance(item, h5py.Dataset):
            print(f"{path} -> shape: {item.shape}, dtype: {item.dtype}")
        elif isinstance(item, h5py.Group):
            print(f"{path}/ (group)")
            walk_h5(item, path)
with h5py.File("../data/example_data.zarr/labels/(3, 5).h5", "r") as f:
    # Segmentation labels: (T, Y, X) or (T, Z, Y, X) format
    label_volume = f["segmentation/images"][:]  # shape: (75, 6048, 6048)
    
    # Track data: We'll use the 'map' (T, ID) and 'LBEPR' (ID metadata) arrays
    track_map = f["tracks/obj_type_1/map"][:]     # shape: (2420, 2)
    track_coords = f["objects/obj_type_1/coords"][:]  # shape: (41424, 5)

# Build a Napari-compatible track array: (track_id, T, Y, X)
# coord columns = [T, Z, Y, X, something?] → we'll ignore Z and take Y, X
# The 'labels' array gives track_id for each coordinate
    track_ids = f["objects/obj_type_1/labels"][:].squeeze()  # shape: (N,)
time = track_coords[:, 0]
y = track_coords[:, 2]
x = track_coords[:, 3]


NameError: name 'np' is not defined

In [21]:
df = pd.read_pickle('/mnt/OPERA2/Nathan/macrohet_syno/results/dfs/sc_df.pkl')

In [27]:
subset_df = df[(df['Experiment ID'] == 'PS0000') 
& (df['Acquisition ID'] == (3, 5))
& (df['mtb_origin'] != 'Junk')]

In [32]:
subset_df['Cell ID'].unique()

array([1, 107, 108, 109, 115, 117, 118, 134, 138, 142, 144, 147, 15, 150,
       161, 170, 174, 178, 186, 187, 19, 194, 196, 197, 210, 217, 218,
       224, 226, 227, 233, 239, 247, 250, 252, 254, 256, 263, 264, 268,
       272, 278, 289, 291, 295, 300, 302, 309, 319, 325, 327, 329, 330,
       331, 334, 335, 336, 337, 339, 341, 345, 352, 355, 36, 363, 366,
       367, 373, 374, 380, 393, 401, 421, 427, 430, 437, 438, 44, 444,
       445, 447, 450, 452, 456, 458, 459, 460, 463, 465, 466, 470, 474,
       475, 477, 481, 483, 484, 486, 487, 490, 492, 495, 501, 503, 507,
       508, 51, 510, 513, 515, 517, 518, 521, 525, 526, 528, 530, 531,
       536, 537, 542, 544, 552, 558, 56, 564, 565, 577, 579, 584, 586,
       598, 601, 602, 607, 608, 610, 617, 624, 665, 670, 69, 702, 77, 802,
       808, 810, 823, 840, 869, 892, 90, 93], dtype=object)

In [34]:
subset_df['x'].max()*5.04

5985.280546875

In [31]:
subset_df[['Cell ID', 'Time (hours)', 'y', 'x']]

,Cell ID,Time (hours),y,x
405,1,0.0,876.779602,519.922607
406,1,1.0,876.766357,522.290833
407,1,2.0,874.563110,524.336243
408,1,3.0,876.656799,516.952454
409,1,4.0,880.909363,521.947449
...,...,...,...,...
1068443,93,70.0,752.825500,524.823853
1068444,93,71.0,751.840698,521.677734
1068445,93,72.0,754.471313,527.788879
1068446,93,73.0,758.731323,532.674438


In [22]:
label_volume.shape

(75, 6048, 6048)

In [37]:
import pandas as pd
import zarr

In [39]:
# Convert DataFrame to tracks format
scaled_tracks = subset_df[["Cell ID", "Time (hours)", "y", "x"]].copy()
scaled_tracks[["y", "x"]] *= 5.04  # scale coordinates
scaled_tracks.columns = ["track_id", "time", "y", "x"]

# Convert to float32 NumPy array
tracks_array = scaled_tracks.to_numpy(dtype=np.float32)

# Save everything to Zarr
z = zarr.open("../data/example_data.zarr", mode="a")

# Write segmentation (overwrite if needed)
z.create_dataset("labels/0", data=label_volume, overwrite=True, compressor=zarr.Blosc(cname="zstd", clevel=5))
z.attrs["labels"] = [{"path": "labels/0", "type": "label"}]

# Write tracks
z.create_dataset("tracks", data=tracks_array, overwrite=True, compressor=zarr.Blosc(cname="zstd", clevel=5))
z.attrs["tracks_metadata"] = {
    "format_version": "0.1",
    "type": "napari_tracks",
    "columns": ["track_id", "time", "y", "x"]
}